In [1]:
import os
from datetime import datetime
import numpy as np
import pandas as pd
from scipy import stats

URL_BASE = 'https://raw.githubusercontent.com/ab-courses/simulator-ab-datasets/main/2022-04-13/'

def read_database(file_name):
    return pd.read_csv(os.path.join(URL_BASE, file_name))

df_web_logs = read_database('2022-04-13T12_df_web_logs.csv')
df_web_logs['date'] = pd.to_datetime(df_web_logs['date'])
df_users = read_database('experiment_users.csv')

begin_date = datetime(2022, 4, 5)
end_date = datetime(2022, 4, 12)
df_metrics = (
    df_web_logs
    [(df_web_logs['date'] >= begin_date) & (df_web_logs['date'] < end_date)]
    [['user_id', 'load_time']]
)
df = pd.merge(df_users, df_metrics, on='user_id', how='left')[['pilot', 'load_time']]

def get_ci_bootstrap_normal(boot_metrics: np.array, pe_metric: float, alpha: float=0.05):
    """Строит нормальный доверительный интервал.

    boot_metrics - значения метрики, полученные с помощью бутстрепа
    pe_metric - точечная оценка метрики
    alpha - уровень значимости
    
    return: (left, right) - границы доверительного интервала.
    """
    c = stats.norm.ppf(1 - alpha / 2)
    se = np.std(boot_metrics)
    left, right = pe_metric - c * se, pe_metric + c * se
    return left, right

B = 1000
alpha = 0.05
values_a = df[df['pilot'] == 0]['load_time'].values
values_b = df[df['pilot'] == 1]['load_time'].values

quantiles = np.array([0.7, 0.74, 0.78, 0.82, 0.86, 0.9, 0.95, 0.99, 0.999, 0.9999])
for q in quantiles:
    pe = np.quantile(values_b, q) - np.quantile(values_a, q)
    bootstrap_values_a = np.random.choice(values_a, (B, len(values_a)), True)
    bootstrap_metrics_a = np.quantile(bootstrap_values_a, q, axis=1)
    bootstrap_values_b = np.random.choice(values_b, (B, len(values_b)), True)
    bootstrap_metrics_b = np.quantile(bootstrap_values_b, q, axis=1)
    bootstrap_stats = bootstrap_metrics_b - bootstrap_metrics_a
    left, right = get_ci_bootstrap_normal(bootstrap_stats, pe, alpha)
    print(f'q = {q}, ci = [{left:0.3f}, {right:0.3f}], is effect = {not(left < 0 < right)} ')


URLError: <urlopen error [Errno 60] Operation timed out>